In [ ]:
!pip install datasets
!pip install accelerate -U

In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel , Trainer,  TrainingArguments, DataCollatorForLanguageModeling
from torch.utils.data import DataLoader
from datasets import load_dataset
import pandas as pd

In [ ]:
model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(model_name)

dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")

small_dataset = dataset.shuffle(seed=42).select(range(1000))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

...

wikitext-2-raw-v1/validation-00000-of-00(…): reconstructing file:   0%|          |  0.00B /  657kB            

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [ ]:
df = pd.DataFrame(small_dataset)
print(df.head())

                                                text
0   Continuous , short @-@ arc , high pressure xe...
1   Field Marshal Antonio José de Sucre is portra...
2   Norman Gary Finkelstein ( born December 8 , 1...
3   Galveston has several state @-@ funded charte...
4   Walpole 's works have not been completely neg...


In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = small_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
df = pd.DataFrame(tokenized_dataset)
print(df.head())

                                           input_ids  \
0  [45012, 837, 1790, 2488, 12, 31, 10389, 837, 1...   
1  [7663, 37899, 11366, 36997, 390, 47352, 260, 3...   
2  [19636, 10936, 4463, 7750, 5714, 357, 4642, 34...   
3  [5027, 4223, 261, 468, 1811, 1181, 2488, 12, 3...   
4  [6445, 36869, 705, 82, 2499, 423, 407, 587, 31...   

                                      attention_mask  
0  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...  
1  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...  
2  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...  
3  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...  
4  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...  


In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

In [ ]:
batch_size = 4
data_loader = DataLoader(tokenized_dataset, batch_size=batch_size, collate_fn=data_collator)

for batch in data_loader:
    break

for key, value in batch.items():
    print(f"{key}: {value}")

input_ids: tensor([[45012,   837,  1790,  2488,    12,    31, 10389,   837,  1029,  3833,
         27132,   261, 10389, 32209,   423,   257,  3124,  5951,  7173,  5561,
         39204, 19613, 19606,   290,   389,   973,   287,  6591,   985, 24325,
           764,  1320,   318,   837,   262, 15358,  1512,   414,   286,   777,
         32209,  7173,  5561, 26748,   257, 16968,  2042,  1767, 50036,   326,
           468,   257,  5951,  1969,   284,   326,  6515,   422,   262,  3825,
           764,  2293,   484,   547,   717,  5495,  1141,   262, 16236,    82,
           837,   777, 32209,  2540, 13586,   262, 12238,  2488,    12,    31,
          5615,  6588, 10389, 32209,   287,  3807,  1628,   669,   764,  1119,
           389,  9322,   287,  7226,  3439,  3020,   837,   314, 22921,   290,
           262,   649,  4875,  1628,   669,  2646, 20128,  3341,   837, 29393,
           367,  2389, 46783,   837,  1029,  2488,    12,    31,   886,   366,
         16106,   366,  7644,  8091,   29

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    # overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    save_steps=1000,
    save_total_limit=2,
    logging_steps=50,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

In [ ]:
trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,3.966165
100,3.725191
150,3.706260
200,3.597398
250,3.541206
300,3.135668
350,3.013187
400,3.205491
450,3.060134
500,3.071804


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=2.894007678222656, metrics={'train_runtime': 210.7974, 'train_samples_per_second': 23.719, 'train_steps_per_second': 5.93, 'total_flos': 326615040000000.0, 'train_loss': 2.894007678222656, 'epoch': 5.0})

In [ ]:
model.save_pretrained("./fine_tuned_gpt2")
tokenizer.save_pretrained("./fine_tuned_gpt2")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./fine_tuned_gpt2/tokenizer_config.json', './fine_tuned_gpt2/tokenizer.json')

In [ ]:
model_name = "./fine_tuned_gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
def generate_text(prompt, model, tokenizer, max_length=50):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(inputs.input_ids, max_length=max_length, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
prompt = "Once upon a time"
generated_text = generate_text(prompt, model, tokenizer)
print(generated_text)

Once upon a time , the British Empire was the most powerful nation in the world . The British Empire was the most powerful nation in the world , and the British Empire was the most powerful nation in the world . 
 
 
  
